In [1]:
# ============================================================
# CELL 1: Setup
# ============================================================
import os
import gc
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, accuracy_score, classification_report

# Working directory
PROJECT_DIR = r"C:\Users\User\Desktop\research\bengali-smishing"
os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Working directory: C:\Users\User\Desktop\research\bengali-smishing
Device: cuda
GPU: NVIDIA GeForce RTX 2050


In [3]:
# ============================================================
# CELL 2: Load BangalaBarta Dataset
# ============================================================
barta_path = "data/external/BangalaBarta bangla_spam_sms smishing.csv"

# চেক করুন file exists
if not os.path.exists(barta_path):
    print(f"❌ File not found: {barta_path}")
    print("Please download from Kaggle and place in data/external/")
    print("\nFiles in data/external/:")
    if os.path.exists("data/external"):
        print(os.listdir("data/external"))
else:
    df_barta = pd.read_csv(barta_path)
    print(f"✅ Loaded: {df_barta.shape}")
    print(f"\nColumns: {df_barta.columns.tolist()}")
    print(f"\nLabel distribution:")
    print(df_barta['label'].value_counts() if 'label' in df_barta.columns else df_barta.iloc[:, -1].value_counts())
    print(f"\nFirst 3 rows:")
    print(df_barta.head(3))

✅ Loaded: (2772, 2)

Columns: ['label', 'text']

Label distribution:
label
smish     924
promo     924
normal    924
Name: count, dtype: int64

First 3 rows:
   label                                               text
0  smish  সোনালী ব্যাংক অ্যাকাউন্টে সমস্যা হয়েছে। কল কর...
1  smish  ক্রিপ্টো বিনিয়োগে লাভবান হন! আজই শুরু করুন: h...
2  promo  স্পেশাল ডিল শেষ দিন,৩০জিবি @৩০০৳,৩০দিন! আজই না...


In [5]:
# ============================================================
# CELL 3 (FIXED): Label Mapping with Normalization
# ============================================================
# Model labels
model_labels = ['normal', 'promo', 'smish']
model_label2id = {l: i for i, l in enumerate(model_labels)}
print(f"Model labels: {model_label2id}")

# ---------- Normalize BangalaBarta labels ----------
# Step 1: strip whitespace + lowercase
df_barta['label_clean'] = df_barta['label'].astype(str).str.strip().str.lower()

print(f"\nAfter normalization:")
print(df_barta['label_clean'].value_counts())

# ---------- Map to model labels ----------
# Different dataset may use different names
barta_label_map = {
    # Standard names (already match)
    'smish': 'smish',
    'promo': 'promo',
    'normal': 'normal',
    # Alternative names
    'smishing': 'smish',
    'promotional': 'promo',
    'legitimate': 'normal',
    'ham': 'normal',
    'spam': 'smish',
}

# Apply mapping
df_barta['label_mapped'] = df_barta['label_clean'].map(barta_label_map)

# Check unmapped
unmapped = df_barta[df_barta['label_mapped'].isna()]
if len(unmapped) > 0:
    print(f"\n⚠️ Unmapped labels:")
    for label in unmapped['label_clean'].unique():
        print(f"   '{label}' — add to barta_label_map")
    print(f"\nUpdate the map and re-run.")
else:
    # Encode
    df_barta['label_encoded'] = df_barta['label_mapped'].map(model_label2id)
    
    print(f"\n✅ All labels mapped successfully")
    print(f"\nFinal label distribution:")
    print(df_barta['label_encoded'].value_counts().sort_index())
    
    # Sanity check
    print(f"\nLabel mapping verified:")
    for i, label in enumerate(model_labels):
        count = (df_barta['label_encoded'] == i).sum()
        print(f"   {label} (id={i}): {count}")

Model labels: {'normal': 0, 'promo': 1, 'smish': 2}

After normalization:
label_clean
smish     924
promo     924
normal    924
Name: count, dtype: int64

✅ All labels mapped successfully

Final label distribution:
label_encoded
0    924
1    924
2    924
Name: count, dtype: int64

Label mapping verified:
   normal (id=0): 924
   promo (id=1): 924
   smish (id=2): 924


In [6]:
# ============================================================
# CELL 4: Load Trained Model (lora_b1_final)
# ============================================================
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

model_path = "results/models/lora_b1_final"
print(f"Loading from: {os.path.abspath(model_path)}")

# Check folder
if not os.path.exists(model_path):
    print(f"❌ Model not found: {model_path}")
    print("\nAvailable models:")
    print(os.listdir("results/models"))
else:
    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
    
    # Base + LoRA
    base_model = AutoModelForSequenceClassification.from_pretrained(
        "xlm-roberta-base", num_labels=3
    )
    model = PeftModel.from_pretrained(base_model, model_path)
    model.to(device)
    model.eval()
    
    print("✅ Model loaded successfully")

Loading from: C:\Users\User\Desktop\research\bengali-smishing\results\models\lora_b1_final


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded successfully


In [7]:
# ============================================================
# CELL 5: Prediction on BangalaBarta
# ============================================================
from torch.utils.data import Dataset, DataLoader
from transformers import DataCollatorWithPadding

class SMSDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True,
            padding="max_length", max_length=self.max_len,
            return_tensors=None
        )
        return enc

# ⚠️ আপনার BangalaBarta-র text column-এর নাম চেক করুন
# যদি 'text' না হয়, উপযুক্ত নাম দিন
text_column = 'text'  # অথবা 'message', 'sms', etc.

texts = df_barta[text_column].astype(str).tolist()
print(f"Total texts: {len(texts)}")
print(f"Sample: {texts[0][:100]}")

# Dataset & DataLoader
test_ds = SMSDataset(texts, tokenizer)
test_loader = DataLoader(
    test_ds, batch_size=32,
    collate_fn=DataCollatorWithPadding(tokenizer)
)

# Predict
all_preds = []
all_probs = []

print("\nRunning predictions...")
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        probs = torch.softmax(outputs.logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

df_barta['predicted'] = all_preds
df_barta['predicted_label'] = df_barta['predicted'].map({i: l for l, i in model_label2id.items()})

print(f"✅ Predictions complete: {len(all_preds)}")

Total texts: 2772
Sample: সোনালী ব্যাংক অ্যাকাউন্টে সমস্যা হয়েছে। কল করুন: +8801818788890

Running predictions...
✅ Predictions complete: 2772


In [8]:
# ============================================================
# CELL 6: Results & Comparison
# ============================================================
# Metrics
y_true = df_barta['label_encoded'].values
y_pred = df_barta['predicted'].values

macro_f1 = f1_score(y_true, y_pred, average='macro')
accuracy = accuracy_score(y_true, y_pred)

# Smish recall
smish_idx = model_label2id['smish']
smish_mask = y_true == smish_idx
smish_recall = (y_pred[smish_mask] == smish_idx).mean()

print("=" * 60)
print("CROSS-DATASET RESULTS: BangalaBarta")
print("=" * 60)
print(f"Total samples : {len(df_barta)}")
print(f"Macro F1      : {macro_f1:.4f}")
print(f"Accuracy      : {accuracy:.4f}")
print(f"Smish Recall  : {smish_recall:.4f}")

print("\n" + "=" * 60)
print("COMPARISON: B1 (In-Distribution) vs BangalaBarta")
print("=" * 60)
print(f"{'Dataset':<25s} | {'F1':>8s} | {'Smish Rec':>10s}")
print("-" * 50)
print(f"{'B1 (in-distribution)':<25s} | {0.744:>8.4f} | {0.591:>10.4f}")
print(f"{'BangalaBarta (cross)':<25s} | {macro_f1:>8.4f} | {smish_recall:>10.4f}")
print("-" * 50)
print(f"{'Drop':<25s} | {0.744-macro_f1:>+8.4f} | {0.591-smish_recall:>+10.4f}")
print("=" * 60)

print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=model_labels,
    labels=[0, 1, 2]
))

CROSS-DATASET RESULTS: BangalaBarta
Total samples : 2772
Macro F1      : 0.8487
Accuracy      : 0.8568
Smish Recall  : 0.6050

COMPARISON: B1 (In-Distribution) vs BangalaBarta
Dataset                   |       F1 |  Smish Rec
--------------------------------------------------
B1 (in-distribution)      |   0.7440 |     0.5910
BangalaBarta (cross)      |   0.8487 |     0.6050
--------------------------------------------------
Drop                      |  -0.1047 |    -0.0140

Classification Report:
              precision    recall  f1-score   support

      normal       0.90      0.98      0.94       924
       promo       0.76      0.98      0.86       924
       smish       0.99      0.60      0.75       924

    accuracy                           0.86      2772
   macro avg       0.88      0.86      0.85      2772
weighted avg       0.88      0.86      0.85      2772



In [9]:
# ============================================================
# CELL 7: Save Results
# ============================================================
import json

cross_dataset_results = {
    "dataset": "BangalaBarta",
    "total_samples": len(df_barta),
    "macro_f1": float(macro_f1),
    "accuracy": float(accuracy),
    "smish_recall": float(smish_recall),
    "b1_f1": 0.744,
    "b1_smish_recall": 0.591,
    "f1_drop": float(0.744 - macro_f1),
    "smish_recall_drop": float(0.591 - smish_recall),
}

os.makedirs("results/analysis", exist_ok=True)
with open("results/analysis/cross_dataset_bangalabarta.json", "w") as f:
    json.dump(cross_dataset_results, f, indent=2)

print("✅ Saved: results/analysis/cross_dataset_bangalabarta.json")
print(f"\nSummary:")
print(json.dumps(cross_dataset_results, indent=2))

✅ Saved: results/analysis/cross_dataset_bangalabarta.json

Summary:
{
  "dataset": "BangalaBarta",
  "total_samples": 2772,
  "macro_f1": 0.8486636083781591,
  "accuracy": 0.8567821067821068,
  "smish_recall": 0.604978354978355,
  "b1_f1": 0.744,
  "b1_smish_recall": 0.591,
  "f1_drop": -0.10466360837815913,
  "smish_recall_drop": -0.013978354978354979
}


In [10]:
# ============================================================
# VERIFY ALL NUMBERS FROM JSON FILES
# ============================================================
import json

print("=" * 70)
print(" " * 20 + "ALL NUMBERS VERIFICATION")
print("=" * 70)

# 1. Main result (B1)
with open("results/final_main_results.json") as f:
    main = json.load(f)
print(f"\n📌 MAIN RESULT (B1):")
print(f"   Macro F1      : {main.get('lora_test_f1', 'N/A')}")
print(f"   Smish Recall  : {main.get('smish_recall', 'N/A')}")
print(f"   Accuracy      : {main.get('lora_test_accuracy', 'N/A')}")

# 2. Robustness
with open("results/analysis/robustness_results.json") as f:
    rob = json.load(f)
print(f"\n📌 ROBUSTNESS (Clean):")
for r in rob['results']:
    if r['condition'] == 'Clean':
        print(f"   Macro F1      : {r['macro_f1']}")
        print(f"   Smish Recall  : {r['smish_recall']}")
        break

# 3. BangalaBarta
with open("results/analysis/cross_dataset_bangalabarta.json") as f:
    barta = json.load(f)
print(f"\n📌 BANGALABARTA (Cross-Dataset):")
print(f"   Macro F1      : {barta.get('macro_f1', 'N/A')}")
print(f"   Smish Recall  : {barta.get('smish_recall', 'N/A')}")
print(f"   Accuracy      : {barta.get('accuracy', 'N/A')}")

# 4. Random vs Unseen
with open("results/analysis/random_vs_unseen.json") as f:
    rvu = json.load(f)
print(f"\n📌 RANDOM vs UNSEEN:")
print(f"   Random F1     : {rvu['random_split']['test_f1']}")
print(f"   Random Recall : {rvu['random_split']['smish_recall']}")
print(f"   Unseen F1     : {rvu['unseen_b1']['test_f1']}")
print(f"   Unseen Recall : {rvu['unseen_b1']['smish_recall']}")

print("\n" + "=" * 70)

                    ALL NUMBERS VERIFICATION

📌 MAIN RESULT (B1):
   Macro F1      : 0.7463
   Smish Recall  : 0.5888
   Accuracy      : 0.7451

📌 ROBUSTNESS (Clean):
   Macro F1      : 0.746337323978222
   Smish Recall  : 0.5887924230465666

📌 BANGALABARTA (Cross-Dataset):
   Macro F1      : 0.8486636083781591
   Smish Recall  : 0.604978354978355
   Accuracy      : 0.8567821067821068

📌 RANDOM vs UNSEEN:
   Random F1     : 0.9553308694177755
   Random Recall : 0.9572953736654805
   Unseen F1     : 0.686678059601794
   Unseen Recall : 0.5193370165745856

